# PII Detection in Agent Traces

This notebook demonstrates how to use the `uaef.security.pii` module to scan agent conversations for Personally Identifiable Information (PII). We'll:

1. Run a Strands agent that handles user queries containing PII
2. Use the PII detector to scan the agent trace for sensitive data
3. Show how to configure detection for different PII types
4. Scan structured dictionaries (like agent trace metadata)

## Setup

In [ ]:
from dotenv import load_dotenv
loaded = load_dotenv()
print(f"Environment loaded: {loaded}")

In [ ]:
from uaef.security.pii import (
    PIIDetector,
    PIIDetectionConfig,
    PIIType,
    PIIMatch,
    detect_pii,
    create_default_detector,
)
print("✓ PII detection module imported")

## 1. Quick Start — Detecting PII in Plain Text

The simplest way to use the module is the `detect_pii` convenience function.

In [ ]:
# Detect PII in a sample string
text = "Please contact John at john.doe@acme.com or call 555-867-5309. His SSN is 123-45-6789."

matches = detect_pii(text)

print(f"Found {len(matches)} PII match(es):\n")
for m in matches:
    print(f"  [{m.pii_type.value:12}] \"{m.text}\" (positions {m.start}-{m.end}, confidence={m.confidence})")

## 2. Create a Strands Agent That Handles Sensitive Data

We'll build a simple customer-support agent that naturally encounters PII in user queries.

In [ ]:
from strands import Agent, tool
from uaef.adapters import StrandsAdapter


@tool
def lookup_account(email: str) -> dict:
    """Look up a customer account by email address."""
    # Simulated database lookup
    accounts = {
        "jane.smith@example.com": {"name": "Jane Smith", "account_id": "ACCT-9921", "status": "active"},
        "bob.jones@corp.net": {"name": "Bob Jones", "account_id": "ACCT-4455", "status": "suspended"},
    }
    return accounts.get(email, {"error": f"No account found for {email}"})


@tool
def check_payment(card_last_four: str) -> dict:
    """Check the payment status for a card ending in the given 4 digits."""
    return {"card_ending": card_last_four, "last_payment": "2026-06-30", "amount": "$49.99", "status": "paid"}


support_agent = Agent(
    tools=[lookup_account, check_payment],
    system_prompt=(
        "You are a customer support assistant. Help users with account lookups and payment inquiries. "
        "Always use the provided tools to look up information."
    ),
)
print("✓ Customer support agent created with tools: lookup_account, check_payment")

### 2.1 Run the agent with a PII-laden query

In [ ]:
# This query intentionally contains PII: email, phone, and a credit card number
user_query = (
    "Hi, my name is Jane Smith. My email is jane.smith@example.com and my phone is (415) 555-0198. "
    "Can you look up my account? Also, I paid with card 4111-1111-1111-1111 last month."
)

support_agent.messages.clear()
result = support_agent(user_query)

print("Agent response:")
print(result)

### 2.2 Convert to canonical trace and scan for PII

In [ ]:
adapter = StrandsAdapter()

trace = adapter.transform_to_canonical({
    "messages": support_agent.messages,
    "metrics_summary": support_agent.event_loop_metrics.get_summary(),
    "stop_reason": result.stop_reason,
    "session_id": "pii_demo_session",
})

print(f"Trace has {len(trace.messages)} messages")
print(f"Input: {trace.messages[0].content[:80]}...")

In [ ]:
# Scan the full conversation for PII
# Use ALL types to catch everything
config = PIIDetectionConfig(enabled_types={PIIType.ALL})
detector = PIIDetector(detection_config=config)

print("=" * 70)
print("PII SCAN RESULTS — Agent Conversation")
print("=" * 70)

# Scan user input
input_matches = detector.detect(trace.input)
print(f"\n📥 User Input ({len(input_matches)} PII found):")
for m in input_matches:
    print(f"   • {m.pii_type.value}: \"{m.text}\"")

# Scan agent output
if trace.output:
    output_matches = detector.detect(trace.output)
    print(f"\n📤 Agent Output ({len(output_matches)} PII found):")
    for m in output_matches:
        print(f"   • {m.pii_type.value}: \"{m.text}\"")

# Scan each message in the trace
print(f"\n💬 Per-message scan:")
for i, msg in enumerate(trace.messages):
    if msg.content:
        try:
            msg_matches = detector.detect(msg.content)
            if msg_matches:
                print(f"   Message {i} ({msg.role}): {len(msg_matches)} PII item(s)")
                for m in msg_matches:
                    print(f"      - {m.pii_type.value}: \"{m.text}\"")
        except ValueError:
            pass  # skip empty messages

## 3. Scanning Structured Data with `scan_dict`

The PII detector can also recursively scan dictionaries — useful for scanning trace metadata, tool call arguments, or evaluation results.

In [ ]:
# Simulate structured data from an agent trace
trace_metadata = {
    "session_id": "sess-001",
    "user_info": {
        "query": "My email is alice@company.org, SSN 987-65-4321",
        "ip_address": "Client connected from 192.168.1.42",
    },
    "tool_calls": [
        {"tool": "lookup_account", "args": "email=bob@example.com"},
        {"tool": "check_payment", "args": "card=4222222222222222"},
    ],
    "response": "Your account is active. No PII here.",
}

results = detector.scan_dict(trace_metadata)

print("Dictionary scan results:")
print("-" * 50)
for key, matches in results.items():
    if isinstance(matches, list):
        print(f"\n  Key: '{key}'")
        for m in matches:
            print(f"    → {m.pii_type.value}: \"{m.text}\"")
    elif isinstance(matches, dict):
        print(f"\n  Key: '{key}' (nested)")
        for sub_key, sub_matches in matches.items():
            print(f"    Sub-key: '{sub_key}'")
            for m in sub_matches:
                print(f"      → {m.pii_type.value}: \"{m.text}\"")

## 4. Configuring Detection — Selective PII Types

You don't always need to scan for everything. Configure the detector to focus on specific PII types.

In [ ]:
# Only detect credit cards and SSNs (high-risk PII)
strict_config = PIIDetectionConfig(
    enabled_types={PIIType.CREDIT_CARD, PIIType.SSN}
)
strict_detector = PIIDetector(detection_config=strict_config)

text = (
    "User jane@mail.com called 555-123-4567. "
    "SSN: 111-22-3333. Card: 5500 0000 0000 0004."
)

matches = strict_detector.detect(text)
print("Strict mode (credit cards + SSNs only):")
for m in matches:
    print(f"  [{m.pii_type.value}] \"{m.text}\"")

print(f"\nEmail and phone were ignored: {not any(m.pii_type == PIIType.EMAIL for m in matches)}")

## 5. Using `has_pii` for Quick Pass/Fail Checks

For gating logic (e.g., deciding whether to store a trace), use the boolean `has_pii` method.

In [ ]:
detector = create_default_detector()

safe_text = "The weather in Seattle is 72°F and sunny."
risky_text = "Please update my email to sarah.connor@skynet.io"

print(f"Safe text has PII:  {detector.has_pii(safe_text)}")
print(f"Risky text has PII: {detector.has_pii(risky_text)}")

# Example gating logic
if detector.has_pii(risky_text):
    print("\n⚠️  PII detected — trace should not be stored without redaction.")
else:
    print("\n✓ No PII — safe to store.")

## 6. Putting It All Together — End-to-End PII Audit

A reusable pattern: run an agent, capture the trace, scan for PII, and produce a summary report.

In [ ]:
def audit_trace_for_pii(trace, detector=None):
    """Scan an AgentTrace for PII and return a summary report."""
    if detector is None:
        config = PIIDetectionConfig(enabled_types={PIIType.ALL})
        detector = PIIDetector(detection_config=config)

    report = {
        "session_id": trace.session_id,
        "total_pii_found": 0,
        "pii_by_type": {},
        "messages_with_pii": [],
    }

    for i, msg in enumerate(trace.messages):
        if not msg.content:
            continue
        try:
            matches = detector.detect(msg.content)
        except ValueError:
            continue

        if matches:
            report["total_pii_found"] += len(matches)
            report["messages_with_pii"].append({
                "index": i,
                "role": msg.role,
                "pii_count": len(matches),
                "types": [m.pii_type.value for m in matches],
            })
            for m in matches:
                report["pii_by_type"][m.pii_type.value] = (
                    report["pii_by_type"].get(m.pii_type.value, 0) + 1
                )

    return report


# Run the audit on our earlier trace
report = audit_trace_for_pii(trace)

print("PII Audit Report")
print("=" * 50)
print(f"Session: {report['session_id']}")
print(f"Total PII items found: {report['total_pii_found']}")
print(f"\nBreakdown by type:")
for pii_type, count in sorted(report["pii_by_type"].items()):
    print(f"  {pii_type}: {count}")
print(f"\nMessages containing PII: {len(report['messages_with_pii'])}")
for msg_info in report["messages_with_pii"]:
    print(f"  Message {msg_info['index']} ({msg_info['role']}): {msg_info['types']}")

## Summary

The `uaef.security.pii` module provides:

| Feature | Usage |
|---------|-------|
| Quick detection | `detect_pii(text)` |
| Configurable types | `PIIDetectionConfig(enabled_types={...})` |
| Boolean check | `detector.has_pii(text)` |
| Dict scanning | `detector.scan_dict(data)` |
| Full detector | `PIIDetector(detection_config=config)` |

**Supported PII types:** email, phone, SSN, credit card, IP address, URL, date of birth.

Use this in your evaluation pipelines to flag traces containing sensitive data before storing or sharing them.